<a href="https://colab.research.google.com/github/women-in-ai-ireland/June-2024-Group-002/blob/embeddings/QA_TEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install langchain -U langchain-community faiss-cpu langchain-openai tiktoken unstructured selenium newspaper3k textstat
!pip install sentence-transformers==2.2.2
!pip install InstructorEmbedding
!pip install langchain langchain-community
!pip install pypdf

In [8]:
import pickle
import os
from langchain.document_loaders import WebBaseLoader, UnstructuredURLLoader, NewsURLLoader, SeleniumURLLoader
from langchain_openai import OpenAI
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader
#from InstructorEmbedding import INSTRUCTOR
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain.chains import RetrievalQA
from pypdf import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')


In [ ]:
os.environ["OPENAI_API_KEY"] = "OPEN AI KEY"


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
root_dir = "/content/gdrive/MyDrive/WAI_project/"


In [ ]:
instructor_embeddings = HuggingFaceInstructEmbeddings(model_name="hkunlp/instructor-xl",
                                                      model_kwargs={"device": "cuda"})


In [ ]:
embedding_store_path = f"{root_dir}/embedding_store"


In [ ]:
def store_embeddings(docs, embeddings, sotre_name, path):
    vectorStore = FAISS.from_documents(docs, embeddings)
    with open(f"{path}/faiss_{sotre_name}.pkl", "wb") as f:
        pickle.dump(vectorStore, f)


In [ ]:
def load_embeddings(sotre_name, path):
    with open(f"{path}/faiss_{sotre_name}.pkl", "rb") as f:
        VectorStore = pickle.load(f)
    return VectorStore


In [ ]:
pdfs = [
    # Add paths to your PDF documents here
    '/content/gdrive/MyDrive/WAI_project/AQA-8035-TG-NEE-CS-STUDENT-BOOKLET.PDF',
    # Add more PDF paths as needed
]

# Using PDF Loader on multiple PDFs
pdf_documents = []
for pdf in pdfs:
    loader = PyPDFLoader(pdf)
    pdf_documents.extend(loader.load())

# Combine documents from website and PDFs
documents = pdf_documents

# Creating the chunking function using Recursive Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100,
    length_function = len,
)

# Chunking
texts = text_splitter.split_documents(documents)
# print(len(texts))
# print(texts[100])


In [ ]:
# Test FAISS using Instructor Embeddings without saving into a file
db_instructEmbedd = FAISS.from_documents(texts, instructor_embeddings)
retriever = db_instructEmbedd.as_retriever(search_kwargs={"k": 3}) # return the 3 closest vectors
docs = retriever.invoke("Who is Fela Kuti?")
docs[0]

# Store embeddings in a local folder
store_embeddings(docs,
                 instructor_embeddings,
                 sotre_name='instructEmbeddings',
                 path=embedding_store_path)

# Load the embeddings
db_instructEmbedd = load_embeddings(sotre_name='instructEmbeddings',
                                    path=embedding_store_path)

# Get embeddings from local db
retriever = db_instructEmbedd.as_retriever(search_kwargs={"k": 3})
docs = retriever.get_relevant_documents("Who is Fela Kuti?")
docs[0]
